# 24 · Cierre definitivo de embeddings: recuperación + Gemma 4 31B

**Propósito:** decidir entre **Qwen3-Embedding-4B**, **PPLX Embed v1 4B** y **Nemotron-3-Embed-1B** para la recuperación de evidencia bibliográfica que alimentará a `gemma4:31b-it-qat`.

**Esta es la última evaluación de selección; no vuelve a calcular los embeddings.** Reutiliza los **519 vectores por modelo** y las **24 búsquedas densas completas** ya existentes. Evalúa **12 consultas en español (N01–N12)** y **6 pares de autores** (3 con coautoría registrada, 3 sin coautoría registrada *en este corpus*) con los mismos criterios para los tres recuperadores: **54 respuestas de Gemma**.

- **Consultas:** recupera los 5 primeros artículos de cada ranking ya calculado; los juicios de pertinencia fueron **asistidos por IA y no validados por humanos**.
- **Pares de autores:** usa un artículo independiente de cada autor como semilla vectorial y recupera otros 2 artículos propios por autor. **Excluye todas sus publicaciones compartidas y los artículos semilla.** Estas pruebas examinan la **justificación documental**, **no la precisión de la predicción de enlaces**.
- **Generación:** mismo Gemma, prompt, contexto máximo, temperatura y límite de respuesta. Gemma **no recibe el nombre del recuperador**, ni los juicios ni la clase de coautoría.
- **Control:** comprueba citas existentes, referencias a artículos de ambos autores, pertinencia anotada y respuestas sin evidencia. **Una cita válida NO prueba que la afirmación sea fiel**; por tanto no se confunde esta auditoría automática con verificación humana de factualidad.

## 1. Cargar y verificar el corpus y los resultados existentes (CPU)

Puedes abrir la libreta desde la raíz del repositorio, `07_RAG` o la carpeta `Benchmark_Externo`. No se descarga nada ni se ocupa la GPU en estas celdas.

In [1]:
import csv, hashlib, json, math, os, re, time
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import numpy as np
import pandas as pd
from IPython.display import display

VERSION = "CIERRE_EMBEDDINGS_GEMMA4_V1"
GEMMA = "gemma4:31b-it-qat"
MODELOS = ["qwen3_4b", "pplx_4b_bf16", "nemotron3_1b"]
NOMBRE = {"qwen3_4b": "Qwen3-Embedding-4B", "pplx_4b_bf16": "PPLX Embed 4B", "nemotron3_1b": "Nemotron-3-Embed-1B"}
ARCHIVOS_NPZ = {"qwen3_4b": "qwen3_4b_vectores_v1.npz", "pplx_4b_bf16": "pplx_4b_bf16_vectores_v1.npz", "nemotron3_1b": "nemotron3_1b_vectores_v1.npz"}
K_DOCUMENTOS, K_POR_AUTOR, MAX_ABSTRACT = 5, 2, 1000
NUM_CTX, NUM_PREDICT, SEMILLA = 4096, 320, 2026
MIN_MEJORA_INTERNA, MIN_CITAS_VALIDAS = 0.03, 0.90

REPO = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "06_grafos/catalogo_articulos.csv").is_file()
             and (p / "07_RAG").is_dir()), None)
if REPO is None:
    raise FileNotFoundError("Abre esta libreta dentro de Tesis_Multimodelo: falta 06_grafos/catalogo_articulos.csv")
RAG = REPO / "07_RAG"
PRUEBAS = RAG / "Pruebas_RAG"
CARPETA = PRUEBAS / "Benchmark_Externo"
RUTAS = {
    "catalogo": REPO / "06_grafos/catalogo_articulos.csv",
    "ranking": PRUEBAS / "densa_resultados_v1.json",
    "qrels": PRUEBAS / "pertinencia_completa_N01_N12.json",
    "externo": CARPETA / "benchmark_finalistas_v2.json",
    **{m: RAG / f for m, f in ARCHIVOS_NPZ.items()}
}
for nombre, p in RUTAS.items():
    if not p.is_file():
        raise FileNotFoundError(f"Falta {nombre}: {p}. No se reemplaza ni vuelve a generar.")

def sha256(p):
    h = hashlib.sha256()
    with p.open("rb") as f:
        for bloque in iter(lambda: f.read(1024 * 1024), b""): h.update(bloque)
    return h.hexdigest()

def ahora(): return datetime.now(timezone.utc).isoformat()
def verificar(ok, mensaje):
    if not ok: raise ValueError(mensaje)

HASH = {k: sha256(v) for k, v in RUTAS.items()}
with RUTAS["catalogo"].open(encoding="utf-8-sig", newline="") as f:
    articulos = list(csv.DictReader(f))
POR_ID = {str(int(a["article_id"])): a for a in articulos}
IDS = list(POR_ID)
verificar(len(articulos) == len(POR_ID) == 519, "Se esperaban 519 artículos únicos")
D = json.loads(RUTAS["ranking"].read_text(encoding="utf-8"))
A = json.loads(RUTAS["qrels"].read_text(encoding="utf-8"))
EX = json.loads(RUTAS["externo"].read_text(encoding="utf-8"))
verificar(D["fuentes"]["sha256_catalogo"] == HASH["catalogo"], "Ranking y catálogo no coinciden")
verificar(EX.get("estado") == "completo", "El benchmark externo no está completo")
verificar(len(A.get("anotaciones", {})) > 1200, "Se requiere pertinencia COMPLETA N01–N12, no el archivo parcial")
verificar(all(m in D["sistemas"] for m in MODELOS), "Falta un modelo entre los rankings completos")
PUNTAJE_EXTERNO = {r["modelo"]: r["ndcg10_macro_ponderado"] for r in EX["comparacion"]["resumen"] if r["modelo"] in MODELOS}
verificar(len(PUNTAJE_EXTERNO) == 3, "Faltan puntuaciones externas")

V = {}
for m in MODELOS:
    with np.load(RUTAS[m], allow_pickle=False) as z:
        verificar(str(z["sha256_catalogo"].item()) == HASH["catalogo"], f"NPZ incompatible: {m}")
        verificar([str(x) for x in z["article_ids"]] == IDS, f"Orden de article_id incompatible: {m}")
        verificar(str(z["modelo_revision"].item()) == EX["protocolo"]["modelos"][m]["revision"], f"Revisión diferente: {m}")
        x = np.ascontiguousarray(z["vectores"], dtype=np.float32)
        verificar(x.shape == (519, EX["protocolo"]["modelos"][m]["dim"]), f"Forma inesperada: {m}")
        verificar(np.isfinite(x).all() and np.allclose(np.linalg.norm(x, axis=1), 1, atol=2e-4), f"Vectores inválidos: {m}")
        V[m] = x
POSICION = {id_: i for i, id_ in enumerate(IDS)}

RANKINGS = {}
for m in MODELOS:
    entradas = D["sistemas"][m]["resultados"]
    verificar(len(entradas) == 24, f"Se esperaban 24 consultas en {m}")
    RANKINGS[m] = {r["query_id"]: [str(int(d["article_id"])) for d in r["documentos"]] for r in entradas}
    verificar(all(len(v) == 519 and set(v) == set(IDS) for v in RANKINGS[m].values()), f"Ranking no exhaustivo: {m}")
CONSULTAS = {r["query_id"]: r["consulta"] for r in D["consultas"]}
verificar(all(f"N{i:02d}-ES" in CONSULTAS for i in range(1, 13)), "Faltan las 12 consultas españolas")
print("Datos válidos: 519 artículos, 3 modelos, 24 rankings y benchmark externo completo.")
print("No se cargaron modelos de embeddings ni se usó la GPU.")

Datos válidos: 519 artículos, 3 modelos, 24 rankings y benchmark externo completo.
No se cargaron modelos de embeddings ni se usó la GPU.


## 2. Fijar los 18 casos y preparar las mismas evidencias por modelo

En los seis pares, la semilla es **el artículo independiente de menor `article_id`** de cada autor; la consulta vectorial es la media normalizada de esas dos semillas en el espacio de cada modelo. Se recuperan **dos artículos diferentes y adicionales de cada autor**, siempre sin publicaciones compartidas. Esto permite contrastar la calidad de las justificaciones con documentos reales sin codificar nuevos textos.

In [2]:
# Grados de pertinencia asistidos. Ausencia de juicio NO se interpreta como irrelevancia verificada.
GRADOS = defaultdict(dict)
for llave, anot in A["anotaciones"].items():
    m = re.fullmatch(r"(N\d\d)-A(\d+)", llave)
    if not m: continue
    info = anot.get("asistida", {})
    if info.get("estado") == "anotada" and isinstance(info.get("grado"), int) and info["grado"] in (0, 1, 2):
        GRADOS[m.group(1)][str(int(m.group(2)))] = info["grado"]
verificar(all(any(g > 0 for g in GRADOS[f"N{i:02d}"].values()) for i in range(1, 13)), "Faltan positivos anotados en alguna necesidad")

def ndcg5(ids, necesidad):
    grados = GRADOS[necesidad]
    real = sum(grados.get(id_, 0) / math.log2(i + 2) for i, id_ in enumerate(ids[:5]))
    ideal = sorted((x for x in grados.values() if x > 0), reverse=True)[:5]
    maximo = sum(g / math.log2(i + 2) for i, g in enumerate(ideal))
    return real / maximo if maximo else 0.0

# Los nombres y las categorías se comprueban contra el catálogo: no se inventan enlaces.
PARES = [
    ("P01", "Boris Escalante Ramírez", "Jorge Luis Pérez González", True),
    ("P02", "Gemma Bel Enguix", "Helena Montserrat Gómez Adorno", True),
    ("P03", "Armando Castañeda Rojano", "Sergio Rajsbaum Gorodezky", True),
    ("P04", "Helena Montserrat Gómez Adorno", "Jimena Olveres Montiel", False),
    ("P05", "Boris Escalante Ramírez", "Nidiyare Hevia Montiel", False),
    ("P06", "Gemma Bel Enguix", "Iván Vladimir Meza Ruiz", False),
]
DOCS_AUTOR = defaultdict(set)
for id_, a in POR_ID.items():
    for autor in [s.strip() for s in a["autores"].split(";") if s.strip()]:
        DOCS_AUTOR[autor].add(id_)

CASOS = []
for i in range(1, 13):
    qid = f"N{i:02d}-ES"; n = qid[:3]
    for modelo in MODELOS:
        docs = RANKINGS[modelo][qid][:K_DOCUMENTOS]
        CASOS.append({"caso": qid, "tipo": "tema", "modelo": modelo,
                       "pregunta": CONSULTAS[qid], "documentos": docs,
                       "ndcg5_asistido": round(ndcg5(docs, n), 8),
                       "cobertura_juzgada5": sum(d in GRADOS[n] for d in docs) / 5})

for pid, a, b, colaboracion_observada in PARES:
    verificar(a in DOCS_AUTOR and b in DOCS_AUTOR, f"Autor no encontrado: {pid}")
    conjuntos = DOCS_AUTOR[a], DOCS_AUTOR[b]
    compartidos = conjuntos[0] & conjuntos[1]
    verificar(bool(compartidos) == colaboracion_observada, f"La coautoría registrada no coincide: {pid}")
    propios_a = sorted(conjuntos[0] - compartidos, key=int)
    propios_b = sorted(conjuntos[1] - compartidos, key=int)
    verificar(min(len(propios_a), len(propios_b)) >= 4, f"No hay suficientes artículos independientes: {pid}")
    sem_a, sem_b = propios_a[0], propios_b[0]
    elegibles_a, elegibles_b = propios_a[1:], propios_b[1:]
    for modelo in MODELOS:
        mat = V[modelo]
        q = mat[POSICION[sem_a]] + mat[POSICION[sem_b]]
        q = q / np.linalg.norm(q)
        def mejores(candidatos):
            return sorted(candidatos, key=lambda x: (-float(mat[POSICION[x]] @ q), int(x)))[:K_POR_AUTOR]
        ra, rb = mejores(elegibles_a), mejores(elegibles_b)
        docs = [ra[0], rb[0], ra[1], rb[1]]  # Alternar autores; nunca incluir semillas ni artículos compartidos
        verificar(len(set(docs)) == 4 and not (set(docs) & compartidos), "Fuga de artículo compartido")
        CASOS.append({"caso": pid, "tipo": "coautoria", "modelo": modelo,
                       "pregunta": f"¿Qué evidencia documental respalda o limita la posible colaboración académica entre {a} y {b}?",
                       "documentos": docs, "autores": [a, b], "semillas_excluidas": [sem_a, sem_b],
                       "compartidos_excluidos": sorted(compartidos, key=int),
                       "coautoria_registrada_en_corpus": colaboracion_observada})

verificar(len(CASOS) == 54 and len(set((r["caso"], r["modelo"]) for r in CASOS)) == 54, "Se esperaban 54 comparaciones")

def evidencia(id_):
    a = POR_ID[id_]
    resumen = a["abstract"]
    aviso = " [Resumen abreviado para el contexto; el embedding se calculó sobre el resumen original íntegro.]" if len(resumen) > MAX_ABSTRACT else ""
    return (f"[A{int(id_):04d}] Título: {a['titulo']}\nAutores: {a['autores']}\n"
            f"Año: {a['anio']} | Área: {a['area']}\n"
            f"Palabras clave: {a['keywords'][:220]}\n"
            f"Resumen: {resumen[:MAX_ABSTRACT]}{aviso}")

print("Casos fijos:", len(CASOS), "(12 temas + 6 pares) × 3 modelos = 54 respuestas")
print("Pares P01–P03: coautoría registrada, TODAS las publicaciones compartidas excluidas del contexto.")
print("Pares P04–P06: sin coautoría registrada en este catálogo; no significa ausencia global de colaboración.")
print("nDCG@5 asistido de las 12 consultas (referencia secundaria, NO evaluación humana):")
display(pd.DataFrame([{"Modelo": NOMBRE[m], "nDCG@5": round(np.mean([x["ndcg5_asistido"] for x in CASOS if x["modelo"] == m and x["tipo"] == "tema"]), 4),
                       "Externo nDCG@10": round(PUNTAJE_EXTERNO[m], 4)} for m in MODELOS]))

Casos fijos: 54 (12 temas + 6 pares) × 3 modelos = 54 respuestas
Pares P01–P03: coautoría registrada, TODAS las publicaciones compartidas excluidas del contexto.
Pares P04–P06: sin coautoría registrada en este catálogo; no significa ausencia global de colaboración.
nDCG@5 asistido de las 12 consultas (referencia secundaria, NO evaluación humana):


,Modelo,nDCG@5,Externo nDCG@10
0,Qwen3-Embedding-4B,0.9290,0.6009
1,PPLX Embed 4B,0.9254,0.5859
2,Nemotron-3-Embed-1B,0.9334,0.5817


## 3. Comprobar Ollama y preparar el resultado con reanudación

Ollama debe estar ejecutándose y el modelo `gemma4:31b-it-qat` **ya descargado**. La libreta no lo descarga automáticamente. No uses Ollama a la vez que otro modelo de GPU. El archivo `.jsonl` es un registro **append-only**, por lo que no se reemplaza el JSON después de cada respuesta (evita el bloqueo que ocurrió con el benchmark previo).

In [3]:
import requests
API = "http://127.0.0.1:11434"
try:
    respuesta = requests.get(API + "/api/tags", timeout=10)
    respuesta.raise_for_status()
    instalados = respuesta.json().get("models", [])
except requests.RequestException as e:
    raise RuntimeError("Ollama no responde. Abre Ollama y vuelve a ejecutar SOLO esta celda.") from e
coincidentes = [m for m in instalados if m.get("name") == GEMMA or m.get("model") == GEMMA]
verificar(coincidentes, f"Falta el modelo local {GEMMA}. Verifica 'ollama list' antes de continuar.")
DIGEST_GEMMA = coincidentes[0].get("digest", "")

SALIDA = CARPETA / "cierre_embeddings_gemma4_v1.jsonl"
PROCEDIMIENTO = {"version": VERSION, "gemma": GEMMA, "gemma_digest": DIGEST_GEMMA,
                 "modelos": MODELOS, "k_documentos": K_DOCUMENTOS, "por_autor": K_POR_AUTOR,
                 "max_abstract_caracteres": MAX_ABSTRACT, "num_ctx": NUM_CTX,
                 "num_predict": NUM_PREDICT, "semilla": SEMILLA,
                 "margen_cambio": MIN_MEJORA_INTERNA, "min_citas_validas": MIN_CITAS_VALIDAS,
                 "hash_archivos": HASH,
                 "casos": CASOS}
FIRMA = hashlib.sha256(json.dumps(PROCEDIMIENTO, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()

def anexar(objeto):
    # Una línea completa por llamada, con flush + fsync, sin .tmp ni os.replace.
    texto = json.dumps(objeto, ensure_ascii=False, allow_nan=False) + "\n"
    for intento in range(5):
        try:
            with SALIDA.open("a", encoding="utf-8") as f:
                f.write(texto)
                f.flush()
                os.fsync(f.fileno())
            return
        except PermissionError:
            if intento == 4: raise
            time.sleep(2 * (intento + 1))

if not SALIDA.exists():
    with SALIDA.open("x", encoding="utf-8") as f:
        f.write(json.dumps({"tipo": "protocolo", "firma": FIRMA, "procedimiento": PROCEDIMIENTO,
                            "inicio_utc": ahora()}, ensure_ascii=False) + "\n")
        f.flush(); os.fsync(f.fileno())

lineas = SALIDA.read_text(encoding="utf-8").splitlines()
verificar(lineas, "Registro vacío")
try:
    anterior = [json.loads(s) for s in lineas]
except json.JSONDecodeError as e:
    raise ValueError("Una línea JSONL quedó incompleta; no se modifica automáticamente el archivo.") from e
verificar(anterior[0].get("tipo") == "protocolo" and anterior[0].get("firma") == FIRMA,
         "La prueba anterior usa otros datos/opciones. Se detiene para no mezclar experimentos.")
HECHOS = {(r["caso"], r["modelo"]): r for r in anterior[1:] if r.get("tipo") == "respuesta"}
verificar(len(HECHOS) == sum(r.get("tipo") == "respuesta" for r in anterior), "Se detectaron respuestas duplicadas")
print("Gemma instalado:", GEMMA, "| digest:", DIGEST_GEMMA[:16])
print("Respuestas recuperadas de archivo:", len(HECHOS), "/ 54")
print("Archivo único de resultados:", SALIDA)

Gemma instalado: gemma4:31b-it-qat | digest: e0812a55773bfeac
Respuestas recuperadas de archivo: 0 / 54
Archivo único de resultados: C:\Users\hazar\Documents\GitHub\Tesis_Multimodelo\07_RAG\Pruebas_RAG\Benchmark_Externo\cierre_embeddings_gemma4_v1.jsonl


## 4. Ejecutar Gemma para los 54 casos (reanudable)

**Esta es la única celda larga.** Guarda cada respuesta y muestra un tiempo estimado basado en las llamadas reales. Puedes detener y ejecutar otra vez esta celda: solo repetirá llamadas que no se hayan guardado.

Gemma recibe **la misma instrucción** y como máximo 5 artículos, pero artículos diferentes según el recuperador. Las respuestas no se califican por un evaluador LLM: se auditan citas, estructura y pertinencia previamente anotada. El tiempo puede ser muy variable porque **Gemma 31B no cabe completamente en 12 GB de VRAM**.

In [4]:
SISTEMA = (
    "Eres un investigador que justifica con cautela afirmaciones a partir de evidencia bibliográfica. "
    "Utiliza EXCLUSIVAMENTE los documentos del contexto. No inventes obras, autores, hallazgos, "
    "vínculos ni resultados. Cada afirmación concreta debe llevar una cita como [A0123]. "
    "Distingue compatibilidad temática de colaboración comprobada o predicción de enlaces. "
    "Si la evidencia es débil, explica el límite. "
    "Responde SOLO un objeto JSON con exactamente las claves: "
    "respuesta (texto de 80-120 palabras con citas [A0001]), "
    "citas (lista de identificadores A0001 usados en respuesta), "
    "limitaciones (una frase), suficiencia (suficiente, parcial o insuficiente)."
)

def instrucciones(caso):
    contexto = "\n\n".join(evidencia(d) for d in caso["documentos"])
    if caso["tipo"] == "coautoria":
        detalle = (f"CASO HIPOTÉTICO. Autores A y B: {caso['autores'][0]} / {caso['autores'][1]}. "
                   "Evalúa afinidad de temas SOLO a partir de los artículos dados. "
                   "Cita al menos un trabajo de cada autor y aclara lo que no prueba la evidencia. "
                   "No se te informa si ya han coescrito artículos, ni existe una predicción de enlace aquí.")
    else:
        detalle = ("Describe brevemente qué aportan los artículos a la pregunta; "
                   "evita afirmar que todos responden directamente si la evidencia no lo permite.")
    return f"PREGUNTA: {caso['pregunta']}\n{detalle}\n\nARTÍCULOS DISPONIBLES:\n{contexto}"

def cita_normalizada(c):
    m = re.fullmatch(r"\[?A0*(\d+)\]?", str(c).strip(), re.IGNORECASE)
    return "A" + f"{int(m.group(1)):04d}" if m else str(c).strip()

def llamar_gemma(caso):
    peticion = {"model": GEMMA, "stream": False, "format": "json", "think": False,
                "keep_alive": "30m", "options": {"temperature": 0, "num_ctx": NUM_CTX,
                                               "num_predict": NUM_PREDICT, "seed": SEMILLA},
                "messages": [{"role": "system", "content": SISTEMA},
                             {"role": "user", "content": instrucciones(caso)}]}
    inicio = time.perf_counter()
    r = requests.post(API + "/api/chat", json=peticion, timeout=900)
    r.raise_for_status()
    datos = r.json()
    verificar(datos.get("done") is True, "Respuesta de Ollama incompleta")
    bruto = datos.get("message", {}).get("content", "")
    try:
        obj = json.loads(bruto)
        correcto = isinstance(obj, dict) and all(k in obj for k in ("respuesta", "citas", "limitaciones", "suficiencia"))
    except (ValueError, TypeError):
        obj, correcto = {}, False
    texto = str(obj.get("respuesta", "")) if correcto else bruto
    declaradas = [cita_normalizada(v) for v in obj.get("citas", [])] if correcto and isinstance(obj.get("citas"), list) else []
    en_texto = [cita_normalizada(x) for x in re.findall(r"\[(A\d+)\]", texto)]
    permitidas = {f"A{int(id_):04d}" for id_ in caso["documentos"]}
    invalidas = sorted(set(declaradas + en_texto) - permitidas)
    concuerdan = bool(en_texto) and set(declaradas) == set(en_texto)
    citados = sorted(set(en_texto) & permitidas)
    por_autor = None
    if caso["tipo"] == "coautoria":
        por_autor = {autor: any(str(int(c[1:])) in DOCS_AUTOR[autor] for c in citados)
                     for autor in caso["autores"]}
    grados_citados = None
    if caso["tipo"] == "tema":
        grados_citados = {c: GRADOS[caso["caso"][:3]].get(str(int(c[1:]))) for c in citados}
    return {"tipo": "respuesta", "caso": caso["caso"], "modelo": caso["modelo"],
            "fecha_utc": ahora(), "pregunta": caso["pregunta"], "documentos": caso["documentos"],
            "salida": obj if correcto else {}, "texto_bruto": bruto, "formato_json": bool(correcto),
            "citas_texto": citados, "citas_declaradas": declaradas,
            "citas_invalidas": invalidas, "citas_coinciden": concuerdan,
            "citas_de_ambos_autores": bool(por_autor) and all(por_autor.values()) if por_autor is not None else None,
            "grados_citados_asistidos": grados_citados,
            "tiempo_pared_s": round(time.perf_counter() - inicio, 3),
            "tiempo_ollama_s": round(datos.get("total_duration", 0) / 1e9, 3),
            "tokens_salida": datos.get("eval_count"), "tokens_prompt": datos.get("prompt_eval_count"),
            "terminacion": datos.get("done_reason", "")}

PENDIENTES = [c for c in CASOS if (c["caso"], c["modelo"]) not in HECHOS]
print(f"Llamadas a Gemma pendientes: {len(PENDIENTES)} de 54. Cada respuesta se guarda de inmediato.")
inicio_ronda = time.perf_counter()
for i, caso in enumerate(PENDIENTES, start=1):
    try:
        registro = llamar_gemma(caso)
    except Exception as e:
        print(f"Se detuvo en {caso['caso']} / {caso['modelo']}. Los casos anteriores siguen guardados.")
        raise RuntimeError(f"Fallo en Gemma: {e}") from e
    anexar(registro)
    HECHOS[(caso["caso"], caso["modelo"])] = registro
    segundos_por_llamada = (time.perf_counter() - inicio_ronda) / i
    eta_h = (len(PENDIENTES) - i) * segundos_por_llamada / 3600
    print(f"{len(HECHOS):2d}/54 | {caso['caso']:>6} | {NOMBRE[caso['modelo']]:<20} "
          f"| {registro['tiempo_pared_s']:.1f} s | falta ≈ {eta_h:.1f} h", flush=True)
print("Generación completa:", len(HECHOS), "/54")

Llamadas a Gemma pendientes: 54 de 54. Cada respuesta se guarda de inmediato.
 1/54 | N01-ES | Qwen3-Embedding-4B   | 490.8 s | falta ≈ 7.2 h
 2/54 | N01-ES | PPLX Embed 4B        | 95.1 s | falta ≈ 4.2 h
 3/54 | N01-ES | Nemotron-3-Embed-1B  | 90.7 s | falta ≈ 3.2 h
 4/54 | N02-ES | Qwen3-Embedding-4B   | 80.1 s | falta ≈ 2.6 h
 5/54 | N02-ES | PPLX Embed 4B        | 80.9 s | falta ≈ 2.3 h
 6/54 | N02-ES | Nemotron-3-Embed-1B  | 87.4 s | falta ≈ 2.1 h
 7/54 | N03-ES | Qwen3-Embedding-4B   | 84.7 s | falta ≈ 1.9 h
 8/54 | N03-ES | PPLX Embed 4B        | 84.9 s | falta ≈ 1.7 h
 9/54 | N03-ES | Nemotron-3-Embed-1B  | 91.3 s | falta ≈ 1.6 h
10/54 | N04-ES | Qwen3-Embedding-4B   | 107.8 s | falta ≈ 1.6 h
11/54 | N04-ES | PPLX Embed 4B        | 76.8 s | falta ≈ 1.5 h
12/54 | N04-ES | Nemotron-3-Embed-1B  | 86.8 s | falta ≈ 1.4 h
13/54 | N05-ES | Qwen3-Embedding-4B   | 76.0 s | falta ≈ 1.3 h
14/54 | N05-ES | PPLX Embed 4B        | 79.2 s | falta ≈ 1.3 h
15/54 | N05-ES | Nemotron-3-Embed-1B  

## 5. Comparación final y decisión operativa

Presenta por separado **nDCG@5 asistido**, benchmark externo y controles estructurales de Gemma. No interpreta las citas como una medida suficiente de fidelidad semántica. El dictamen se establece **antes de mirar** las respuestas: preferir Qwen3 por el benchmark externo salvo una mejora interna clara de otro modelo con calidad formal comparable.

Si `formato_json`, las citas o la presencia de ambos autores fallan de forma sustancial, la conclusión quedará señalada como **requiere revisión**, no se inventará evidencia de calidad.

In [5]:
verificar(len(HECHOS) == 54, f"Faltan {54 - len(HECHOS)} respuestas; vuelve a ejecutar solo la celda 4")
FILAS = []
for m in MODELOS:
    rr = [HECHOS[(c["caso"], m)] for c in CASOS if c["modelo"] == m]
    temas = [c for c in CASOS if c["modelo"] == m and c["tipo"] == "tema"]
    pares = [r for r in rr if r["caso"].startswith("P")]
    valido = lambda r: (r["formato_json"] and not r["citas_invalidas"] and r["citas_coinciden"])
    FILAS.append({"clave": m, "Modelo": NOMBRE[m],
                  "Externo nDCG@10": round(PUNTAJE_EXTERNO[m], 4),
                  "Interno asistido nDCG@5": round(float(np.mean([c["ndcg5_asistido"] for c in temas])), 4),
                  "Citas y JSON correctos": round(sum(valido(r) for r in rr) / len(rr), 3),
                  "Pares con citas de ambos": round(sum(bool(r["citas_de_ambos_autores"]) for r in pares) / 6, 3),
                  "Duración Gemma (min)": round(sum(r["tiempo_pared_s"] for r in rr) / 60, 1),
                  "Respuestas": len(rr)})
TABLA = pd.DataFrame(FILAS)
display(TABLA.drop(columns=["clave"]).sort_values("Externo nDCG@10", ascending=False))

qwen = next(r for r in FILAS if r["clave"] == "qwen3_4b")
mayor_interno = max(FILAS, key=lambda r: r["Interno asistido nDCG@5"])
aceptables = [r for r in FILAS if r["Citas y JSON correctos"] >= MIN_CITAS_VALIDAS and r["Pares con citas de ambos"] >= 4/6]
if not aceptables:
    ganador = "qwen3_4b"
    estado = "provisional: los tres generadores presentan problemas estructurales que deben describirse"
elif qwen in aceptables and mayor_interno["Interno asistido nDCG@5"] - qwen["Interno asistido nDCG@5"] < MIN_MEJORA_INTERNA:
    ganador = "qwen3_4b"
    estado = "elección operativa cerrada por benchmark externo y diferencias internas menores al margen prefijado"
else:
    candidatos = [r for r in aceptables if r["Interno asistido nDCG@5"] >= qwen["Interno asistido nDCG@5"] + MIN_MEJORA_INTERNA]
    ganador = max(candidatos, key=lambda r: (r["Interno asistido nDCG@5"], r["Externo nDCG@10"]))["clave"] if candidatos else max(aceptables, key=lambda r: r["Externo nDCG@10"])["clave"]
    estado = "elección operativa por desempeño y controles preestablecidos; revisar excepciones cualitativas"

print("\nMODELO RECOMENDADO PARA IMPLEMENTAR:", NOMBRE[ganador])
print("ESTADO:", estado)
print("CRITERIO: externo como evidencia independiente; interno = grados asistidos, no verdad absoluta.")
print("ADVERTENCIA: no se ha demostrado fidelidad de las explicaciones con jueces humanos ni predicción de enlaces.")

CIERRE = {"tipo": "cierre", "fecha_utc": ahora(), "firma": FIRMA,
          "seleccion": ganador, "estado": estado, "comparacion": FILAS,
          "observaciones": ["No se reentrenaron embeddings ni se realizó ninguna nueva inferencia de embeddings.",
              "12 consultas existentes y 6 casos documentales de coautoría; 54 respuestas de Gemma.",
              "Juicios asistidos por IA, sin validación humana; documentos sin juicio no son falsos.",
              "Coautoría registrada o no registrada SOLO dentro del catálogo de 519 artículos; no se midió predicción.",
              "Citas y JSON correctos no garantizan que todas las afirmaciones estén apoyadas semánticamente.",
              "Decisión operativa reproducible, no superioridad universal ni conclusión estadística nueva."]}
# Se añade un cierre solo si no existía ya la misma versión y contenido.
cierres = [r for r in anterior[1:] if r.get("tipo") == "cierre"]
if not cierres or cierres[-1].get("seleccion") != ganador or cierres[-1].get("comparacion") != FILAS:
    anexar(CIERRE)
print("\nRESULTADO ARCHIVADO:", SALIDA)
print("Conserva la libreta y el JSONL. La siguiente fase es implementar el RAG con el modelo elegido.")

,Modelo,Externo nDCG@10,Interno asistido nDCG@5,Citas y JSON correctos,Pares con citas de ambos,Duración Gemma (min),Respuestas
0,Qwen3-Embedding-4B,0.6009,0.9290,0.944,1.000,33.9,18
1,PPLX Embed 4B,0.5859,0.9254,0.778,0.667,26.4,18
2,Nemotron-3-Embed-1B,0.5817,0.9334,0.944,0.833,27.3,18



MODELO RECOMENDADO PARA IMPLEMENTAR: Qwen3-Embedding-4B
ESTADO: elección operativa cerrada por benchmark externo y diferencias internas menores al margen prefijado
CRITERIO: externo como evidencia independiente; interno = grados asistidos, no verdad absoluta.
ADVERTENCIA: no se ha demostrado fidelidad de las explicaciones con jueces humanos ni predicción de enlaces.

RESULTADO ARCHIVADO: C:\Users\hazar\Documents\GitHub\Tesis_Multimodelo\07_RAG\Pruebas_RAG\Benchmark_Externo\cierre_embeddings_gemma4_v1.jsonl
Conserva la libreta y el JSONL. La siguiente fase es implementar el RAG con el modelo elegido.
